# Deepgram으로 오디오 파일을 전사하고 Anthropic으로 인터뷰 질문 준비하기!

**이 notebook을 자신의 drive에 복사하고 아래 지침을 따르세요!** 🥳🥳🥳

----------------------------

# 시작하기:
다음 세 cell을 실행하면 원하는 오디오를 전사할 수 있습니다. 아래 주석은 원하는 대로 output을 수정하기 위해 조정할 수 있는 변수를 알려줍니다.

이 notebook을 실행하기 전에 전사할 오디오 URL 몇 개를 준비해야 합니다. 원하는 오디오 파일을 사용할 수 있습니다.

그리고 아직 Deepgram에 가입하지 않았다면 다음 링크를 확인하세요: https://dpgr.am/prerecorded-notebook-signup

# 1단계: Dependency

이 cell을 실행하여 필요한 모든 dependency를 다운로드하세요.

참고: 왼쪽의 재생 버튼을 클릭하거나 cell을 클릭한 뒤 `shift`+`ENTER`를 동시에 눌러 cell을 실행할 수 있습니다. (Mac에서는 `shift` + `return`.)

In [ ]:
%pip install requests ffmpeg-python
%pip install deepgram-sdk --upgrade
%pip install requests
%pip install anthropic

# 2단계: 오디오 URL 파일

이 notebook을 사용할 수 있도록 서버에서 호스팅되는 오디오 파일을 찾으세요. 또는 아래 코드에서 Deepgram이 제공하는 예제 파일을 사용하세요. 

In [ ]:
# 위의 2단계를 완료했나요? 👀
# 왼쪽 폴더에 오디오 파일이 보이나요? 📂

# 3단계: 전사

다음 변수를 입력하세요:


* `DG_KEY` = 개인 Deepgram API key
* `AUDIO_FILE_URL` = 전사하려는 오디오 파일의 URL.


이제 cell을 실행하세요! (`Shift` + `Enter`)

-----------



그리고 이미 Deepgram 사용자인데 이 cell에서 오류가 발생한다면 가장 일반적인 해결 방법은 다음과 같습니다:

1. 설치된 deepgram-sdk를 업데이트해야 할 수 있습니다.
2. Deepgram 계정의 남은 credit을 확인해야 할 수 있습니다.

In [ ]:
import requests
from deepgram import DeepgramClient, FileSource, PrerecordedOptions

# Deepgram API key
DG_KEY = "🔑🔑🔑 Your API Key here! 🔑🔑🔑"

# 오디오 파일의 URL
AUDIO_FILE_URL = "https://static.deepgram.com/examples/nasa-spacewalk-interview.wav"

# 전사 JSON 파일을 저장할 경로
TRANSCRIPT_FILE = "transcript.json"


def main():
    try:
        # 1단계: API key로 Deepgram client 생성
        deepgram = DeepgramClient(DG_KEY)

        # URL에서 오디오 파일 다운로드
        response = requests.get(AUDIO_FILE_URL, timeout=60)
        if response.status_code == 200:
            buffer_data = response.content
        else:
            print("Failed to download audio file")
            return

        payload: FileSource = {
            "buffer": buffer_data,
        }

        # 2단계: 오디오 분석을 위한 Deepgram 옵션 설정
        options = PrerecordedOptions(
            model="nova-2",
            smart_format=True,
        )

        # 3단계: 텍스트 payload와 옵션으로 transcribe_file method 호출
        response = deepgram.listen.prerecorded.v("1").transcribe_file(payload, options)

        # 4단계: 응답 JSON을 파일에 기록
        with open(TRANSCRIPT_FILE, "w") as transcript_file:
            transcript_file.write(response.to_json(indent=4))

        print("Transcript JSON file generated successfully.")

    except Exception as e:
        print(f"Exception: {e}")


if __name__ == "__main__":
    main()

위 cell이 성공하면 content 디렉터리에 JSON output 파일이 나타나야 합니다. 참고: cell 실행이 끝난 시점과 JSON 파일이 실제로 나타나는 시점 사이에 약간의 지연이 있을 수 있습니다. 정상적인 현상입니다. 파일이 나타날 때까지 잠시 기다리세요.

# 4단계: 전사 결과 확인

아래 함수는 output JSON을 parsing하고 방금 전사한 파일 중 하나의 전사 결과를 출력합니다! (확인하려는
파일이 실제로 content 디렉터리에 이미 로드되어 있는지 확인하세요.)

**`OUTPUT` 변수를 전사 결과를 확인하려는 파일 이름으로 설정하세요.**

그런 다음 이 cell을 실행하여(`Shift`+`Enter`) 오디오의 문장별 전사 결과를 확인하세요!

In [ ]:
import json

# 읽으려는 output 파일의 경로로 이 변수를 설정
OUTPUT = "transcript.json"


# JSON에는 많은 정보가 있지만 전사 결과만 읽으려면
# 아래 코드를 실행하세요!
def print_transcript(transcription_file):
    with open(transcription_file) as file:
        data = json.load(file)
        result = data["results"]["channels"][0]["alternatives"][0]["transcript"]
        result = result.split(".")
        for sentence in result:
            print(sentence + ".")


print_transcript(OUTPUT)


위 cell이 성공하면 오디오 전사의 일반 텍스트 버전이 나타나야 합니다. 

# 5단계: Anthropic으로 인터뷰 질문 준비하기

이제 인터뷰 질문 준비에 도움이 되는 분석을 위해 전사 결과를 Anthropic에 보낼 수 있습니다. 아래 cell을 실행하여(`Shift`+`Enter`) 위 오디오 전사 결과를 바탕으로 Anthropic이 제안하는 인터뷰 질문을 받으세요.

In [ ]:
import json

import anthropic

transcription_file = "transcript.json"


# JSON 파일에서 전사 결과를 가져오는 함수
def get_transcript(transcription_file):
    with open(transcription_file) as file:
        data = json.load(file)
        result = data["results"]["channels"][0]["alternatives"][0]["transcript"]
        return result


# JSON 파일에서 전사 결과 로드
message_text = get_transcript(transcription_file)

# Claude API client 초기화
client = anthropic.Anthropic(
    # 기본값은 os.environ.get("ANTHROPIC_API_KEY")
    # Claude API key
    api_key="🔑🔑🔑 Your API Key here! 🔑🔑🔑"
)

# API 요청용 텍스트 준비
formatted_messages = [{"role": "user", "content": message_text}]

# 깊이 있는 개방형 인터뷰 질문 생성
response = client.messages.create(
    model="claude-opus-4-8",
    max_tokens=1000,
    temperature=0.5,
    system="Your task is to generate a series of thoughtful, open-ended questions for an interview based on the given context. The questions should be designed to elicit insightful and detailed responses from the interviewee, allowing them to showcase their knowledge, experience, and critical thinking skills. Avoid yes/no questions or those with obvious answers. Instead, focus on questions that encourage reflection, self-assessment, and the sharing of specific examples or anecdotes.",
    messages=formatted_messages,
)

# 생성한 질문 출력

# 각 TextBlock의 텍스트를 하나의 문자열로 결합
content = "".join(block.text for block in response.content)

# 내용을 '\n\n'으로 분리
parts = content.split("\n\n")

# 각 부분에 추가 줄바꿈을 넣어 출력
for part in parts:
    print(part)
    print("\n")

이 cell이 성공하면 원본 오디오 파일에 기반한 인터뷰 질문 목록이 나타나야 합니다. 이제 Deepgram으로 오디오를 전사하고 Anthropic으로 인터뷰 질문을 받을 수 있습니다. 